# Preparation for the analysis

In [44]:
import analysis_functions
import learning_models
import numpy as np
import pandas as pd
import os

import matplotlib.pyplot as plt
import seaborn as sns # Loads in the theme

pd.set_option('future.no_silent_downcasting', True)
#pd.set_option('display.max_rows', None)

In [45]:
# Make output folder
output_plot_dir = os.path.join('..','plots','behavioural_pilot')
if not os.path.exists(output_plot_dir):
    os.makedirs(output_plot_dir)

output_file_dir = os.path.join('..','pre-processed')
if not os.path.exists(output_file_dir):
    os.makedirs(output_file_dir)

In [46]:
# Preparation
experiment_output_path = '/Users/kenneth.van.der.zee/Documents/phd_1_task/experiment_output'

In [47]:
unique_subject_ids = [1, 2, 3]  #list(range(1, 46))
unique_sessions = [1]

recent_results = analysis_functions.list_files_with_date_and_subject_id(experiment_output_path, unique_subject_ids, unique_sessions)

# These are the files the function 'list_files_with_date_and_subject_id' found
print(recent_results)

['/Users/kenneth.van.der.zee/Documents/phd_1_task/experiment_output/experiment_output_sub-001_session-01_2024-04-10_13:37:11.csv', '/Users/kenneth.van.der.zee/Documents/phd_1_task/experiment_output/experiment_output_sub-002_session-01_2024-04-10_15:48:45.csv', '/Users/kenneth.van.der.zee/Documents/phd_1_task/experiment_output/experiment_output_sub-003_session-01_2024-04-12_15:40:55.csv']


In [48]:
# Opens all the files and combines them into one dataframe
combined_results_dataframe = analysis_functions.combine_result_files(recent_results)
print(combined_results_dataframe)

      trial  emotional_cue_time response objectively_correct  \
0         0        1.712749e+09     down                Even   
1         1        1.712749e+09       up                Even   
2         2        1.712749e+09       up                Even   
3         3        1.712749e+09       up                Even   
4         4        1.712749e+09      NaN                Even   
...     ...                 ...      ...                 ...   
1435    475        1.712933e+09       up                True   
1436    476        1.712933e+09     down               False   
1437    477        1.712933e+09     down                True   
1438    478        1.712933e+09       up                True   
1439    479        1.712933e+09       up                True   

      subjectively_correct  response_time      RT_s  stimuli_type  iti  \
0                    False   1.712749e+09  0.947126             3  3.5   
1                    False   1.712749e+09  0.889028             2  3.5   
2        

# WSLS
Determine how often the Win-Stay Lost-Shift strategy is used.
Exports the dataframe afterwards for further analysis

In [49]:
# Add congruency as a column
combined_results_dataframe['congruency'] = combined_results_dataframe.apply(analysis_functions.check_congruency, axis=1)

In [50]:
# Add two 'one-back' columns and a WSLS column
combined_results_WSLS = learning_models.add_WSLS_column(combined_results_dataframe)
print(combined_results_WSLS[['subjectively_correct_one_back','response','response_one_back','WSLS']])

     subjectively_correct_one_back response response_one_back  WSLS
0                              NaN       up               NaN     0
1                             True       up                up     1
2                            False       up                up    -1
3                             True       up                up     1
4                             True       up                up     1
...                            ...      ...               ...   ...
1435                         False     down              down    -1
1436                         False     down              down    -1
1437                         False     down              down    -1
1438                          True     down              down     1
1439                          True       up              down    -1

[1440 rows x 4 columns]


## Separate reversals

In [51]:
combined_results_reversals = analysis_functions.separate_reversals(combined_results_WSLS)
print(combined_results_reversals[['probability_condition','all_switches','all_reversals']])

      probability_condition  all_switches  all_reversals
0                      20.0          True            1.0
1                      20.0         False            0.0
2                      20.0         False            0.0
3                      20.0         False            0.0
4                      20.0         False            0.0
...                     ...           ...            ...
1075                   80.0         False            0.0
1076                   80.0         False            0.0
1077                   80.0         False            0.0
1078                   80.0         False            0.0
1079                   80.0         False            0.0

[1080 rows x 3 columns]


## Variable preparation

In [52]:
mapping = {'up': 1, 'down': 0}
old_column = 'response'
new_column = 'response_integer'
combined_results_reversals = combined_results_reversals[combined_results_reversals['response'].notna()]

combined_results_reversals = analysis_functions.replace_strings_with_integers(combined_results_reversals,
                                                                              mapping,
                                                                              old_column,
                                                                              new_column)

# Export

In [53]:
# Export the whole table as a csv for further analysis
combined_results_reversals.to_csv(os.path.join(output_file_dir,'combined_results_modelling.csv'), index=False, sep=';')